# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Harriet Yayra Boven Fiahagbe
**Student ID:** 59722028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user",   "content": user_prompt},
      ],
      temperature=temperature,
      max_tokens=max_tokens,
  )
  print(f"This is the token usage is: {response.usage}")
  return response.choices[0].message.content

answer = ask_llm("What is the capital of Ghana?")
print(answer)


# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

This is the token usage is: CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.17205964, prompt_time=0.002370422, completion_time=0.013316746, total_time=0.015687168)
The capital of Ghana is Accra.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*


*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. A system role is one that sets the AI's behavior persona, rules and regulations or even boundaries for the entire conversation. It is a background instruction that the AI follows throughout ,without the user never seeing that chat message directly. An example could be: "Give only one sentence or  one word". A user role is the actual input or request which is a specific question or task that is given to the chat box or model. An example could be "Summarize this business contract for me in simple sentences."

2. A token is described as a piece of text (which could be a whole word itself, a part of a word and etc,) where the model breaks text down into as its basic unit of processing. Here the text get broken down into pieces of text (which are the tokens) and each token is then represented as a number for the model to be able to process. Companies like Groq bill based on each token because the actual computational cost  goes up with how much text is processed. If a user  sends a message that is just one word compared to a whole paragraph, the one word would costs the provider less than that of the paragraph, therefore if the billing were per-request its becomes unfair, in the sense that short message may  overpay, while long ones may underpay.

### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra"

print("TEMPERATURE = 0")
for i in range(5):
  answer = ask_llm(question, temperature=0.0)
  print(f"Run {i+1}: {answer}\n")


print("TEMPERATURE = 1.2")
for i in range(5):
  answer = ask_llm(question, temperature=1.2)
  print(f"Run {i+1}: {answer}\n")

TEMPERATURE = 0
This is the token usage is: CompletionUsage(completion_tokens=318, prompt_tokens=55, total_tokens=373, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.173222315, prompt_time=0.001703017, completion_time=0.905689565, total_time=0.907392582)
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Saver**: "Sika" is the Ghanaian word for "money", so this name is straightforward and easy to understand.
6. **

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

For each temperature, beginning with temperature = 0.0, most of the responses given were very much consistent most of the time, yet they were not perfectly identical every single time because there were some little and tiny differences, which creates some  confusion since we know that temperature = 0.0 should be fully determinitic. Nonetheless temperature = 1.2 produced different outputs on every single time that the model ran the question over the five times. The variability is clearly seen all the responses that have been given.

I believe for the load decision support system i am about to build, i would use the lower temperature regime, in order to get responses that are as consistent as possible, since the main aim of the model is to extract facts and trustworthy information from the loan letters. The model giving me consistent data and information would make confident that information  extracted form the letter is accurate and reliable, rather than varying unpredictably.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this"

v1_L002 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}")
v1_L006 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}")

print("\n V1  -  LOO2 \n")
print(v1_L002)
print("\n V1  -  LOO6 \n")
print(v1_L006)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.


SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer in Ghana
                      Summarize loan applications factually and neutrally.
                      Do not invent any details.
                      Keep the summary to 3-4 sentences.
                      """
def summarize_v2(letter_text):
  user_prompt = f"Summarize this loan application:\n\n{letter_text}"
  return ask_llm(user_prompt, system_prompt=SUMMARY_PROMPT_V2, temperature= 0.0)

v2_L002 = summarize_v2(LETTERS['L002'])
v2_L006 = summarize_v2(LETTERS['L006'])


print("\n V1  -  LOO2 \n")
print(v2_L002)
print("\n V1  -  LOO6 \n")
print(v2_L006)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n----SIDE BY SIDE COMPARISON---\n")
print("LOO2: ")
print(f"V1: {v1_L002}\n")
print(f"V2: {v2_L002}\n")


print("\nLOO6: ")
print(f"V1: {v1_L006}\n")
print(f"V2: {v2_L006}\n")

This is the token usage is: CompletionUsage(completion_tokens=69, prompt_tokens=133, total_tokens=202, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.17450999, prompt_time=0.006739728, completion_time=0.190512966, total_time=0.197252694)
This is the token usage is: CompletionUsage(completion_tokens=84, prompt_tokens=135, total_tokens=219, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.064458803, prompt_time=0.038693146, completion_time=0.270018433, total_time=0.308711579)

 V1  -  LOO2 

Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.

 V1  -  LOO6 

Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a p

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1'S Output has some problems that V2 could fix. V1 has some of its sentences paraphrase with some vocabulary changes as compared to the actual letters's wording.  In the original sentences Kwame said  he "can pay back whenever the money comes" yet V1 output he  "promises to repay the loan as soon as possible" subtly changing the meaning because there were no promises or vow that were said in the original statement. Also  this output (v1) had no constraints on it sentence length meaning there is no guarantee that the output would be consistent or be a scannable summary length. In addition V1 used the default 0.7 temperature we had from the  original function created, introducing some unnecessary randomness into the task. Nonetheless for V2 the sentences were closer to the actual letters's wording and structures. The sentences were clean and neutral stating some facts directly from the letter text, ensuring  the values are consistent. An example is in letter six, where Kofi states a fact that his friends claim he is business minded, that he has no collateral  yet he is trustworthy. In  the response summary V2 output  "Kofi claims to be "business minded" based on feedback from friends. He proposes to repay the loan within one year, without offering any collateral, citing his trustworthiness as assurance"


2. The "no invested details" is important, because it prevent the model from adding  information that is not stated in the letter which would make the summary output more inaccurate and unreliable. This means if a loan officer were to trust a summary that  has "invented details" in them, the summary could be regarded as fabricated. In the long run, any decision that could be made from this summary could surely be based on false information. The failure mode in LL literature is known as "Hallucination". This is where a model output or generate confident but false content, rather than stating its uncertainty or fully sticking to the material it had.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0


EXTRACT_PROMPT = """"You are a data extraction assitant for a microfinance loan officer.
                     Extract the following fields from a loan application letter and return only a valid
                     JSON object with exactly these keys, nothing else and no additional explanation

                     applicant_name (string)
                     amount_ghs (number)
                     purpose (string)
                     monthly_profit_ghs (number or null)
                     has_collateral_or_guarantor (boolean)
                     repayment_months (number or null)

                     If a fiels is not stated or seen in the letter use nul. Do not do any guessing or investion of values


                     Example:
                     Letter: "My name is Harriet Boven. I rn a small Abele Walls business and i need GHS 80000 to buy a new fridge.
                     I make about GHS1000 profit a month. My sisters will be my guarantors. I can relap over 8 months."

                    Output:
                    {"applicant_name": "Harriet Boven", "amount_ghs": 8000, "purpose": "to buy a new fridge",
                    "monthly_profit_ghs": 1000, "has_collateral_or_guarantor": true, "repayment_months": 8}
                    """

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

import json

def extract_fields(letter_text):
    user_prompt = f"Extract the fields from this loan application:\n\n{letter_text}"
    first_output = ask_llm(user_prompt, system_prompt=EXTRACT_PROMPT, temperature=0.0)
    cleaned = first_output.strip()

    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Warning: failed to parse JSON for this letter. Raw output:\n{first_output}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

results = []
for letter_id, letter_text in LETTERS.items():
  fields = extract_fields(letter_text)
  if fields is not None:
    fields["letter_id"] = letter_id
    results.append(fields)

import pandas as pd
extraction_final = pd.DataFrame(results)
extraction_final = extraction_final[["letter_id", "applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]]
extraction_final

This is the token usage is: CompletionUsage(completion_tokens=66, prompt_tokens=424, total_tokens=490, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171710296, prompt_time=0.021492693, completion_time=0.097345062, total_time=0.118837755)
This is the token usage is: CompletionUsage(completion_tokens=65, prompt_tokens=384, total_tokens=449, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.064160356, prompt_time=0.043633493, completion_time=0.110893624, total_time=0.154527117)
This is the token usage is: CompletionUsage(completion_tokens=70, prompt_tokens=438, total_tokens=508, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171660311, prompt_time=0.042639407, completion_time=0.118598326, total_time=0.161237733)
This is the token usage is: CompletionUsage(completion_tokens=62, prompt_tokens=404, total_tokens=466, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.17148304, prompt_time=0.02020

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,to buy a deep freezer and expand into frozen f...,900.0,True,20.0
1,L002,Kwame Boateng,25000,to repair my trotro engine and settle some per...,NaN,False,NaN
2,L003,Efua Darko,15000,to purchase two industrial sewing machines and...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,to buy a bulk order of yarn directly from the ...,NaN,True,16.0
5,L006,Kofi,50000,"to start a car washing business, a provision s...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. This is because if one of the examples came from the Letters, the model would be able to see the answer of one of the six letters we are about to test on. This in the long run can contaminate and affect the evaluation badly since we would be essentially leaking the correct output into the prompt for a letter we are trying to extract fields from fairly. Using the monthly_profit_ghs means the model is likely to invent an estimated number or copy a random number into this field.

2. Without the "not null, do not guess" the model would have fabricated and manufactured its own  information. In other words it would have hallucinated by giving it uncertain and unreliable values for that specific column which tends to be the montly_profit_ghs. The model would have  either invented some unseen number into this column or incorrectly copied a different number from about letter into this field.

3. Temperature 0 is the right one for this extraction, because the model to directly extract and pulls facts consistently  from the  main letters. This means there are no going to be any changes or slight differences  or variations in extracted vocabularies. Everything is going to be as factual and neutral as possible, since the values and information is taken directly from the letters with no modification. This is seen in Part 1.2 when we run the questions with a higher and  a  lower temperature. With the lower temperature we could see that the information gotten was very reliable and accurate which is a trait we need in this extraction.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """You are a data extraction assitant for a microfinance loan officer.
                  Given a loan application letter and its respective extracted structured data, produce a decision-support
                  brief with exactly these four sections, remember not to invent any details:

                  - Strengths (bullet points, grounded in the letter)
                  - Risks / red flags (bullet points)
                  - Missing information the officer should request
                  - Suggested next step (e.g. "invite for interview", "request documents" "flag for senior review"). Do NOT "approve" or "reject" under any circumtnaces.

                  Note: You are a decision-support tool ONLY. The final leading decision is always made by a human loan officer, not by you"""
def generate_brief(letter_text, extracted_fields):
  user_prompt = f"""Letter: {letter_text}

                 Extracted data:
                 #This is convert the pyton dicts back into a visible nice formatted Json string
                 {json.dumps(extracted_fields, indent=2)}

                 produce the decision-support brief."""

  return ask_llm(user_prompt, system_prompt=BRIEF_PROMPT, temperature=0.0)

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

briefs = {}
for row in extraction_final.to_dict(orient="records"):
  letter_id = row["letter_id"]
  briefs[letter_id] = generate_brief(LETTERS[letter_id], row)


print("\n---LOO1 BRIEF---\n")
print(briefs["L001"])
print("\n---LOO2 BRIEF---\n")
print(briefs["L002"])
print("\n---LOO3 BRIEF---\n")
print(briefs["L003"])
print("\n---LOO6 BRIEF---\n")
print(briefs["L006"])

This is the token usage is: CompletionUsage(completion_tokens=328, prompt_tokens=428, total_tokens=756, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.175075997, prompt_time=0.047570531, completion_time=1.098552884, total_time=1.146123415)
This is the token usage is: CompletionUsage(completion_tokens=317, prompt_tokens=383, total_tokens=700, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172096276, prompt_time=0.020214393, completion_time=1.050180098, total_time=1.070394491)
This is the token usage is: CompletionUsage(completion_tokens=326, prompt_tokens=446, total_tokens=772, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171875651, prompt_time=0.022991681, completion_time=0.906929694, total_time=0.929921375)
This is the token usage is: CompletionUsage(completion_tokens=341, prompt_tokens=404, total_tokens=745, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171079331, prompt_time=0.

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. Comparing L003 and L006 which are the strongest and weakest letters respectively i believe the system identified the right strengths and red flags in each of them. For the L003 the system was able to list some factual strengths including, Efua's business being a registered business, a clear purchase plan which has been timed to the Christmas season, her consistent monthly profit, her collateral and an 18 months space for payment. Her risks were also made clear which were weather the evaluations of the payment plan over the 15 months and the amount she is loaning based on the festive season.
On the other hand, for L006, the brief was to identify the strength of including, his young and vibrant energy, the fact that his friends see him to be business-minded and his clear vision for businesses and ventures. The brief was also able to correctly identify the application weaknesses, including Kofi having no experience, no collateral, an unrealistic payment plan (based of if the business would boom or not) and a self-asserting report of being trustworthy.

2. We forbade the model from outputting "approve"/"reject" because practically the model also gave its assumptions and contribution from the single letter that was being provided with no access to history credit of the persona, bank records and other documents. This means if the model should have rejected or approved the conclusion would have be based on incomplete information leading to poor outcomes. Ethically, making the model make such decision could have a deadly real world consequences for the applicants and their lives. This is because the model cannot be held accountable in anyway. Having a human loan officer to make this final decision with the LLM serving as a support system,  its the best route to go to in this case.



### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** af1dfda944bf16b4d264c6cb6158ea4fb627376e

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [11]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

#Checking if two values match
def checking_match(gold_value, extracted_value):
  if isinstance(gold_value,str) and isinstance(extracted_value, str):
    return gold_value.strip().lower() == extracted_value.strip().lower()
  return gold_value == extracted_value

fields_to_check = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]

gold_letters = list(GOLD.keys())

extracted_lookup = {row["letter_id"]: row for row in extraction_final.to_dict(orient="records")}

comparison_rows = []
#Looping through each of the fields we are evaluating
for field in fields_to_check:
  result_row = {"field": field}
  correct_count = 0

#Checking this field for each of the 3 gold-labeled letters
  for letter_id in gold_letters:
    gold_value = GOLD[letter_id][field]
    extracted_value = extracted_lookup[letter_id][field]
    matched_value = checking_match(gold_value, extracted_value)

    result_row[letter_id] = matched_value
    correct_count += matched_value

#Storing accuracy as "correct out of total" for this field
  result_row["accuracy"] = f"{correct_count}/{len(gold_letters)}"
  comparison_rows.append(result_row)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
#Turning our results into a readable table
accuracy_df = pd.DataFrame(comparison_rows)
accuracy_df



,field,L001,L003,L006,accuracy
0,applicant_name,True,True,True,3/3
1,amount_ghs,True,True,True,3/3
2,purpose,False,False,False,0/3
3,monthly_profit_ghs,True,True,False,2/3
4,has_collateral_or_guarantor,True,True,True,3/3
5,repayment_months,True,True,True,3/3


### Part 4.2 — Reliability: is the system consistent?

In [15]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def run_reliability_test(letter_text, temperature, no_runs = 5):
  results = []
  for i in range(no_runs):
    user_prompt = f"Extract the fields from this loan application: \n\n{letter_text}"
    raw_output = ask_llm(user_prompt, system_prompt= EXTRACT_PROMPT, temperature=temperature)

    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
      cleaned = cleaned.strip("`")
      cleaned = cleaned.replace("json", "", 1).strip()

    try:
      parsed = json.loads(cleaned)
      results.append(parsed)
    except json.JSONDecodeError:
      results.append(None)
  return results

results_temperature0 = run_reliability_test(LETTERS["L004"], temperature=0)
results_temperature1 = run_reliability_test(LETTERS["L004"], temperature=1.0)

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
def analyze_reliability(results, label):
  correct_count = sum( 1 for r in results if r is not None)
  json_strings = [json.dumps(r, sort_keys= True) for r in results if r is not None]
  special_count = len(set(json_strings))

  #Print outputs
  print(f"--- {label} ---")
  print(f"Valid JSON: {correct_count}/5")
  print(f"Special outputs: {special_count}/{correct_count}")
  print()

analyze_reliability(results_temperature0, "Temperature = 0")
analyze_reliability(results_temperature1, "Temperature = 1.0")



This is the token usage is: CompletionUsage(completion_tokens=62, prompt_tokens=405, total_tokens=467, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.062573978, prompt_time=0.06110946, completion_time=0.102134811, total_time=0.163244271)
This is the token usage is: CompletionUsage(completion_tokens=62, prompt_tokens=405, total_tokens=467, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172308316, prompt_time=0.020688893, completion_time=0.101531299, total_time=0.122220192)
This is the token usage is: CompletionUsage(completion_tokens=62, prompt_tokens=405, total_tokens=467, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172238904, prompt_time=0.021736166, completion_time=0.116576386, total_time=0.138312552)
This is the token usage is: CompletionUsage(completion_tokens=62, prompt_tokens=405, total_tokens=467, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.17188558, prompt_time=0.020467

### Part 4.3 — Hallucination probing

In [19]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
print("--- TEST 1: Aksing our summarize about a detail NOT in the letter ---\n")
test1_prompt = f"""Based on this loan application letter, what is the applicant's credit score?
                   Letter: {LETTERS['L001']}"""
test1_response = ask_llm(test1_prompt, system_prompt= """You are an assistant to a micro finace loan officer. Make sure you only answer
                                                      based on the information explicitly in the letter.
                                                      If something is not stated, say so clearly.""", temperature = 0)


print(test1_response)

#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?
print("--- TEST 2: Feeding an irrelevant text into the extractor ---\n")
weather_reporting = """Today's weather in Cape Coast: sunny with a high of 36°C, low of 22°C.
                       Heavy winds from the southwest at 10km/h. Humidity is around 50%.
                       Rain expected on Thursday """
test2_response = extract_fields(weather_reporting)
print(test2_response)

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("\n--- TEST RESULT ---\n")

print("TEST 1 - Question about a detail NOT in the letter which is the credit score.")
print(f"Output: {test1_response}")
print("Result: PASS - our model correctly states that the credit score is not stated in the letter, rather than inventing one.\n")

print("TEST 2 - Irrelevant text fed into the extractor.")
print(f"Output: {test2_response}")
print("Result: PASS - our model correctly returned none for every field instead of fabricating a fake applicant from the irrelevant text.")

--- TEST 1: Aksing our summarize about a detail NOT in the letter ---

This is the token usage is: CompletionUsage(completion_tokens=13, prompt_tokens=219, total_tokens=232, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.171263195, prompt_time=0.011056523, completion_time=0.030455771, total_time=0.041512294)
The applicant's credit score is not stated in the letter.
--- TEST 2: Feeding an irrelevant text into the extractor ---

This is the token usage is: CompletionUsage(completion_tokens=47, prompt_tokens=344, total_tokens=391, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.062092125, prompt_time=0.027581236, completion_time=0.080334999, total_time=0.107916235)
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}

--- TEST RESULT ---

TEST 1 - Question about a detail NOT in the letter which is the credit score.
Output: The applicant's c

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. From our extraction accuracy report, fields of applicant_name, amount_ghs, has_collaoteral_or_guarantor and repaymet_months, made an accuracy score of 3/3, meaning the model was able to extract these correctly across al the three gold letters  . The montly_profit_ghs information was only given by L001 and L003 making the accuracy 2/3. However the purpose scored  0/3, which is the lowest among all the fields.
From what we have listed above, we can conclude that the hardest field for the model was purpose. This wasnt not because the model failed to extra the purpose. But instead the main problem was that the "purpose" is a free-text field, and the model's wording never matched the gold labels' specific phrasing. Since the accuracy check uses exact string matching, even a correct equivalent extraction  get marked and flagged wrong.

2. For the reliability experiment we got both temperature being 0 and 1.0 to produce 5/5 valid JSON and only 1/5 unique output, which means all five runs taken were identical in both cases. This showed that for a narrow well-defined extraction task, high temperatures does not necessarily hurt reliability as much as you would expect from a creative-writing task which was seen in Part 1.2, where higher temperature produced very different product-name suggestions on the same prompt. Although relying on temperature being 0 is the still the most safe thing to do and more defensible for a production system, since it guarantees determinism rather than depending on the task happening to be "safe" at way higher temperatures.

3. No, the system did not hallucinate in any of the test taken. When we asked about a detail that was not  stated in the letter, the model  correctly responded that the information was not stated, rather than inventing  a fake number. In addition when we fed a completely irrelevant text into the extractor, the model returned NONE for every field instead of fabricating a fake applicant information. The experiment shows that when we have explicit instructions built into the prompts like "based on the information explicitly in the letter" etc. they becomes effective at preventing hallucination, even under inputs specifically designed to tempt the model into inventing its own information.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. Applicants who may write in poor or broken english could be unfairly harmed by our system even if the model does not invent false information by hallucinating. This is because our system relies on how clearly an applicant speaks about their business in writing, therefore even if a successful established business owner write in an informal way in english the system is very much likely to misinterpret it. The system may then flag the unclear statements as weakness to the business, which lowers the possibility of the strong applicant's brief as compared to a weaker business who may have better writing skills. Therefore we can conclude that if the bank makes their system a fully automated decision-making  model, it is going to put applicants for have less formal education as a disadvantage  which is highly unfair.

2. What I would  check before deploying this at a real Ghanaian Microfinance institution is:
i) Is to see if Ghana  has a data protection regulations when it comes to protecting and governing citizen's personal and financial information can be handled, specifically sending the data to a foreign server or company else where.

ii)I would also make sure the customers or my clients has given us an informed consent to their information being processed by an AI system. This will help them understand their letter may pass through some third party channels in order to help the institution process application faster and efficiently.

iii)I would also check the API's provider actual data retention policies and whether they store submitted data, for how long do they keep it, are there risks of the information being leaked and whether it could be used for training their models, with the institution knowledge or consent.
All of these and among other would be thoroughly checked before deploying any AI-assisted tool like this at a real Microfinance business.

3. One concrete safeguard I  would build around this system in production is that there would be a mandatory human review point before any negative or declining outcome reaches the applicant. This is to make sure the model's output is checked for accuracy and fairness before it has any real impact on the client.
Also another safeguard I would look at is having a process of ongoing monitoring where periodically, we would be re-checking the systems's accuracy and its output again new gold-labeled samples over tome. This is is because a model would silently change or alter its behavior and without ongoing monitoring there could go unnoticed, affect the output's accuracy  the model may be bring to the clients and the institution as a whole.





---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1. I think iterating on a prompt is similar to iterating on model hyperparameters as seen in lab three, in that i believe that both follow the same loop and process. They hypothesize, test, measure the result and compare against previous attempts and finally refine! In lab 3, i tried different learning rate and then compared their loss or accuracy curves. In Lab 4, i tried different prompts version including the SUMMARY_PROMPT V1 vs V2 and later compared their outputs side by side as well, keeping the earlier versions for reference. However, the main difference here is what is being changed and how success is measured. In Lab 3, i kept on adjusting the numeric values within a fixed defined mathematical structure and evaluated success quantitatively using the precise metrics like loss and accuracy. While in this current lab, i was rewriting natural language instructions. Evaluating this success was much more qualitative, since i had to read the output given to critically spot little and tiny problems rather than comparing numbers.

2. Basing this decision purely on the technical results from Part 4, since there were some strong extraction accuracy on the structured field, a high reliability even at a higher temperature and no hallucination under the adversarial probing, i would have had a higher chance to trusting this system's technical performance. Yet i do not think i would trust it to run fully unattended to because of the fairness and ethical concerns that i identified in part 4.4. The risk that applicants may not be treated fairly due to the way they speak or write can be classified as highly unethical and truly unfair. From part 4, i realized that most of my answers were correct and valid not because of the technicalities but because of the fairness findings i attached to it which showed that sometimes even technically accurate and reliable systems can still produce wrong and unfair output for certain clients, and this  is exactly the risk that requires human knowledge and collaboration rather than full automation.

3. Processing one loan application required three separate LLM calls which include summarization, extraction and brief generation. Now, based on my own "response.usage" numbers, a typical summarization call would use around 270 tokens, extraction would use about 467 tokens and a brief generation would use around 750 tokens which in total they are about 1487 tokens per application in total. Which means, if we should scale this to a 1000 application per month, it gives us approximately 1487000 tokens per month. In using the model, this is a modest volume by LLM API standards which is a free tier like Groq. Groq can comfortably handle millions of tokens a month, which is reason why it was recommended for this lab. At this scale, cost of the tokens is not the limiting factor. But if the institution scaled more than 10,000 applications a month, a cheaper per token pricing would matter more, and rate limits could become a bigger constraint and problem than raw token cost.

4. In looking back at the course, and the different tasks we have done in each lab, I believe calling an API beats training your own model for a task like this because the LLM was already pretrained on massive amounts of text and already knows language, reasoning, and general world knowledge. Based on this, what I only had to do was to direct that strength of the model with a very good prompt, rather than build the capability from scratch. Having only six loan letters was not enough  to train a model to summarize, extract structured data, and reason about applications from zero. This contrasts with Lab 2 and Lab 3, which required thousands of labeled rows for a single narrow classification task.
However, i think training your own model would make more sense in the opposite situation where there is plenty of labeled data for a narrow and well-defined, task like that of Lab 2/3's obesity classification. Here, when the system needs to run very fast, cheaply, and at a large scale without ongoing per-call cost, or when data privacy and control matter enough, sending information to a third-party API becomes unacceptable and unethical which i directly spoke about in my part 4.4.Overall i believe APIs are good for genera, flexible and language-heavy tasks with little available data while training your own model is better off to narrow, high-volume or privacy-sensitive tasks where you already have data to support it.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.